<a href="https://colab.research.google.com/github/joyashre/LLM-NLP-Learning-task/blob/main/NLP_Task_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 0. Install all required dependencies (Updated)
!pip install -q chromadb bs4 sentence-transformers transformers torch langchain langchain-community langchain-text-splitters

In [2]:
# Experiment 3: RAG QA on Webpages (Flan-T5, direct generate, no pipeline)
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# 1. Load webpage
loader = WebBaseLoader(
    web_paths=("https://en.wikipedia.org/wiki/Natural_language_processing",),
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(id="bodyContent")),
    requests_kwargs={'headers': {'User-Agent': 'Mozilla/5.0'}}
)
docs = loader.load()

# 2. Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# 3. Create vector store
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)

# 4. Load Flan-T5 model (small)
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def rag_query(question):
    # Retrieve relevant context
    retrieved_docs = vectorstore.similarity_search(question, k=3)
    context = "\n".join([doc.page_content for doc in retrieved_docs])

    # Create prompt
    prompt = f"Answer the question based on the context.\nContext: {context}\nQuestion: {question}\nAnswer:"

    # Tokenize and generate
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=100,
        do_sample=False,
        num_beams=1
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

# 5. Ask
question = "What are the main tasks in natural language processing?"
answer = rag_query(question)
print(f"Question: {question}\nAnswer: {answer}")

# Cleanup
vectorstore.delete_collection()

/tmp/ipykernel_12501/1359941511.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warn

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Question: What are the main tasks in natural language processing?
Answer: NER
